# GNN Cloud Notebook: GCN-3 Ensemble Multi-Split

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and validates the plain `GCN-3` ensemble across multiple train/test splits.

This notebook keeps the following improvements fixed:
- Adam with the two-phase schedule
- weighted edges from `adjacency_area.csv`
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling
- standardized regression targets during training
- 3-layer GCN

Validation strategy in this notebook:
- build one ensemble per split seed
- keep the ensemble member seeds fixed inside each split
- aggregate the ensemble results across multiple split seeds
- use this to estimate whether ensemble performance remains strong when the train/validation/test partition changes

This is the correct follow-up if the goal is to verify whether the fixed-split ensemble result generalizes across partitions.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')
        


In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')
        


In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_multi_split'
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
GITHUB_TOKEN = ''
GIT_BRANCH = 'main'
GIT_COMMIT_USERNAME = ''
GIT_COMMIT_EMAIL = ''
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/gcn3_ensemble_multi_split'
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
gnn_root = pipeline_root / 'gnn_prototype'
candidate_module_dirs = [
    gnn_root,
    gnn_root / 'GCN_Optimization',
]
module_dir = None
for candidate in candidate_module_dirs:
    if (candidate / 'colab_gnn_stiffness_prototype.py').is_file():
        module_dir = candidate
        break
if module_dir is None:
    raise FileNotFoundError(f'Could not locate colab_gnn_stiffness_prototype.py under {gnn_root}')

if (gnn_root / 'GCN_Optimization' / 'ensemble_runner.py').is_file():
    optimization_dir = gnn_root / 'GCN_Optimization'
elif (module_dir / 'ensemble_runner.py').is_file():
    optimization_dir = module_dir
else:
    raise FileNotFoundError(f'Could not locate ensemble_runner.py under {gnn_root}')

os.chdir(pipeline_root)

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/gcn3_ensemble_multi_split') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs' / 'gcn3_ensemble_multi_split'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_root.mkdir(parents=True, exist_ok=True)

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'GNN root: {gnn_root}')
print(f'Module dir: {module_dir}')
print(f'Optimization dir: {optimization_dir}')
print(f'Working directory: {Path.cwd()}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')


In [ ]:
import json
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from ensemble_runner import MultiSplitEnsembleConfig, run_multi_split_ensemble


In [ ]:
MEMBER_SEEDS = (11, 42, 73, 101, 202)
SPLIT_SEEDS = (11, 42, 73, 101, 202)
HIDDEN_DIM = 24
DROPOUT = 0.10
WEIGHT_DECAY = 1e-5
LR_PHASE1 = 0.003
LR_PHASE2 = 0.0005
LOSS_NAME = 'mse'
ARCHITECTURE_NAME = 'gcn3'
ARCHITECTURE_LABEL = 'GCN-3'

print(f'Member seeds: {MEMBER_SEEDS}')
print(f'Split seeds: {SPLIT_SEEDS}')
print(f'Hidden dim: {HIDDEN_DIM}')
print(f'Dropout: {DROPOUT}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'LR phase 1: {LR_PHASE1}')
print(f'LR phase 2: {LR_PHASE2}')
print(f'Loss: {LOSS_NAME}')
print(f'Architecture: {ARCHITECTURE_LABEL}')


In [ ]:
config = MultiSplitEnsembleConfig(
    architecture_name=ARCHITECTURE_NAME,
    architecture_label=ARCHITECTURE_LABEL,
    member_seeds=MEMBER_SEEDS,
    split_seeds=SPLIT_SEEDS,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    weight_decay=WEIGHT_DECAY,
    lr_phase1=LR_PHASE1,
    lr_phase2=LR_PHASE2,
    loss_name=LOSS_NAME,
    output_group='gcn3_ensemble_multi_split',
)

result = run_multi_split_ensemble(
    config,
    train_root=train_root,
    predict_root=predict_root,
    output_root=output_root,
)

output_dir = result['output_dir']
split_results = result['split_results']
summary_frame = result['summary_frame']
aggregate_frame = result['aggregate_frame']
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

display(summary_frame)
display(aggregate_frame)


In [ ]:
print(f'Multi-split run dir: {output_dir}')
print('Saved files:')
for file_name in sorted(path.name for path in output_dir.iterdir() if path.is_file()):
    print(f' - {file_name}')

print('Per-split ensemble directories:')
for split_seed, split_result in split_results.items():
    print(f' - split {split_seed}: {split_result["output_dir"]}')


In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODEL_TO_GITHUB:
        for model_path in git_run_dir.rglob('lattice_gnn_model.pt'):
            model_path.unlink()

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add GCN-3 ensemble multi-split cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)


In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_gcn3_ensemble_multi_split.zip'
    !cd /content && zip -qr gnn_outputs_gcn3_ensemble_multi_split.zip gnn_outputs/gcn3_ensemble_multi_split
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')
